# Dry Bean Variety Classification

**AML-MDS project proposal — working notebook**  
**Authors:** Zheshuo Lin
**Date:** 21 October 2026

## Research question

How accurately can seven dry bean varieties be classified from 16 measurements of bean size and shape, as measured by multiclass log loss?

The practical motivation is automatic seed sorting. The dataset provides measurements already extracted from images, so the project can focus on supervised classification using tabular data.

## Dataset and provenance

- **Dataset:** Dry Bean, UCI Machine Learning Repository, dataset ID 602.
- **Source:** https://archive.ics.uci.edu/dataset/602/dry+bean
- **Dataset DOI:** https://doi.org/10.24432/C50S4B
- **License:** Creative Commons Attribution 4.0 International (CC BY 4.0).
- **Unit of observation:** one bean grain.
- **Target:** `Class`, identifying one of seven bean varieties.
- **Inputs:** 16 numerical measurements, including area, perimeter, axis lengths, and shape descriptors.

The source documentation reports 13,611 observations and no missing values. These statements should be checked against the downloaded data in the next section of the analysis.

## 1. Imports and reproducibility

Select this project's Python environment as the notebook kernel. If needed, install the starter dependencies using the commented command below. Set the seed here once and reuse `SEED` in later randomized operations.

In [1]:
# The introduction and data-loading cells were prepared with OpenAI Codex
# (GPT-6), 2026-10-21, adapting the UCI example at the authors' request.
# Optional: uncomment and run once if the packages are missing.
# %pip install ucimlrepo pandas numpy ipykernel

from importlib.metadata import version
from pathlib import Path
import platform

import numpy as np
import pandas as pd
from IPython.display import display
from ucimlrepo import fetch_ucirepo

SEED = 2026

print("Python:", platform.python_version())
for package in ["ucimlrepo", "pandas", "numpy", "ipykernel"]:
    print(f"{package}: {version(package)}")

Python: 3.14.0
ucimlrepo: 0.0.7
pandas: 3.0.6
numpy: 2.5.3
ipykernel: 7.4.0


## 2. Fetch the dataset

Retrieve the dataset and its variable descriptions directly from UCI. This cell needs an internet connection. Keep the original table in `df`; `X` contains the source-defined inputs and `y` contains the class labels. No rows or input columns are removed here.

In [2]:
dry_bean = fetch_ucirepo(id=602)

# Raw table and source-defined feature/target objects.
df = dry_bean.data.original.copy()
X = dry_bean.data.features.copy()
y = dry_bean.data.targets.squeeze("columns").copy()

print("Dataset:", dry_bean.metadata.name)
print("Full table shape:", df.shape)
print("Feature table shape:", X.shape)
print("Target:", y.name)
print("Number of classes:", y.nunique())

display(df.head())

Dataset: Dry Bean
Full table shape: (13611, 17)
Feature table shape: (13611, 16)
Target: Class
Number of classes: 7


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


## 3. Source variable descriptions

Review the source's variable roles and descriptions before interpreting the columns.

In [3]:
display(dry_bean.variables)

,name,role,type,demographic,description,units,missing_values
0,Area,Feature,Integer,None,The area of a bean zone and the number of pixe...,pixels,no
1,Perimeter,Feature,Continuous,None,Bean circumference is defined as the length of...,NaN,no
2,MajorAxisLength,Feature,Continuous,None,The distance between the ends of the longest l...,NaN,no
3,MinorAxisLength,Feature,Continuous,None,The longest line that can be drawn from the be...,NaN,no
4,AspectRatio,Feature,Continuous,None,Defines the relationship between MajorAxisLeng...,NaN,no
5,Eccentricity,Feature,Continuous,None,Eccentricity of the ellipse having the same mo...,NaN,no
6,ConvexArea,Feature,Integer,None,Number of pixels in the smallest convex polygo...,NaN,no
7,EquivDiameter,Feature,Continuous,None,Equivalent diameter: The diameter of a circle ...,NaN,no
8,Extent,Feature,Continuous,None,The ratio of the pixels in the bounding box to...,NaN,no
9,Solidity,Feature,Continuous,None,Also known as convexity. The ratio of the pixe...,NaN,no


## 4. Save a local raw snapshot

Save the downloaded table for later reuse. Run this notebook from the project root so the snapshot is written into `data/`.

In [4]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

raw_data_path = DATA_DIR / "dry_bean_raw.csv"
df.to_csv(raw_data_path, index=False)
print("Raw snapshot saved to:", raw_data_path.resolve())

Raw snapshot saved to: D:\AML\data\dry_bean_raw.csv


## 5. Data Inspections


In [4]:
print("Number of observations (n):", X.shape[0])
print("Number of features (d):", X.shape[1])

print("\nFeature data types:")
display(X.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(pd.DataFrame({
    "missing_n": X.isna().sum(),
    "missing_pct": (X.isna().mean() * 100).round(2)
}))

print("\nDuplicated rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
display(pd.DataFrame({
    "count": y.value_counts(),
    "proportion": y.value_counts(normalize=True).round(4)
}))

Number of observations (n): 13611
Number of features (d): 16

Feature data types:


,dtype
Area,int64
Perimeter,float64
MajorAxisLength,float64
MinorAxisLength,float64
AspectRatio,float64
Eccentricity,float64
ConvexArea,int64
EquivDiameter,float64
Extent,float64
Solidity,float64



Missing values:


,missing_n,missing_pct
Area,0,0.0
Perimeter,0,0.0
MajorAxisLength,0,0.0
MinorAxisLength,0,0.0
AspectRatio,0,0.0
Eccentricity,0,0.0
ConvexArea,0,0.0
EquivDiameter,0,0.0
Extent,0,0.0
Solidity,0,0.0



Duplicated rows:
68

Target distribution:


,count,proportion
Class,,
DERMASON,3546,0.2605
SIRA,2636,0.1937
SEKER,2027,0.1489
HOROZ,1928,0.1417
CALI,1630,0.1198
BARBUNYA,1322,0.0971
BOMBAY,522,0.0384


In [6]:
# Inspect duplicated complete observations
duplicates = df[df.duplicated(keep=False)]

print("Rows involved in duplicates:", len(duplicates))
display(duplicates.sort_values(list(df.columns)).head(20))
conflicting = (
    df.groupby(list(X.columns))["Class"]
      .nunique()
)

print(
    "Feature-identical groups with different labels:",
    (conflicting > 1).sum()
)

Rows involved in duplicates: 136


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
5504,33518,702.956,277.571399,154.305581,1.798842,0.831240,34023,206.582775,0.808383,0.985157,0.852377,0.744251,0.008281,0.001567,0.553909,0.996396,HOROZ
5505,33518,702.956,277.571399,154.305581,1.798842,0.831240,34023,206.582775,0.808383,0.985157,0.852377,0.744251,0.008281,0.001567,0.553909,0.996396,HOROZ
5508,33954,716.750,277.368480,156.356326,1.773951,0.825970,34420,207.922042,0.799482,0.986461,0.830549,0.749624,0.008169,0.001591,0.561936,0.996847,HOROZ
5509,33954,716.750,277.368480,156.356326,1.773951,0.825970,34420,207.922042,0.799482,0.986461,0.830549,0.749624,0.008169,0.001591,0.561936,0.996847,HOROZ
5547,38427,756.323,306.533886,160.591784,1.908777,0.851782,38773,221.193978,0.796976,0.991076,0.844174,0.721597,0.007977,0.001334,0.520702,0.993905,HOROZ
5548,38427,756.323,306.533886,160.591784,1.908777,0.851782,38773,221.193978,0.796976,0.991076,0.844174,0.721597,0.007977,0.001334,0.520702,0.993905,HOROZ
5553,38891,791.343,319.499996,156.869619,2.036723,0.871168,39651,222.525412,0.650025,0.980833,0.780422,0.696480,0.008215,0.001192,0.485085,0.987983,HOROZ
5554,38891,791.343,319.499996,156.869619,2.036723,0.871168,39651,222.525412,0.650025,0.980833,0.780422,0.696480,0.008215,0.001192,0.485085,0.987983,HOROZ
5598,40804,790.802,323.475648,163.287717,1.981016,0.863241,41636,227.932592,0.787570,0.980017,0.819931,0.704636,0.007928,0.001206,0.496512,0.983598,HOROZ
5599,40804,790.802,323.475648,163.287717,1.981016,0.863241,41636,227.932592,0.787570,0.980017,0.819931,0.704636,0.007928,0.001206,0.496512,0.983598,HOROZ


Feature-identical groups with different labels: 0


In [7]:
# Numerical summary
summary = X.describe().T
display(summary)

,count,mean,std,min,25%,50%,75%,max
Area,13611.0,53048.284549,29324.095717,20420.000000,36328.000000,44652.000000,61332.000000,254616.000000
Perimeter,13611.0,855.283459,214.289696,524.736000,703.523500,794.941000,977.213000,1985.370000
MajorAxisLength,13611.0,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860154
MinorAxisLength,13611.0,202.270714,44.970091,122.512653,175.848170,192.431733,217.031741,460.198497
AspectRatio,13611.0,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
Eccentricity,13611.0,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
ConvexArea,13611.0,53768.200206,29774.915817,20684.000000,36714.500000,45178.000000,62294.000000,263261.000000
EquivDiameter,13611.0,253.064220,59.177120,161.243764,215.068003,238.438026,279.446467,569.374358
Extent,13611.0,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195
Solidity,13611.0,0.987143,0.004660,0.919246,0.985670,0.988283,0.990013,0.994677


In [8]:
import numpy as np

print(
    "Non-finite values:",
    (~np.isfinite(X)).sum().sum()
)

Non-finite values: 0


## Use of AI assistants

OpenAI Codex (GPT-6) helped draft this notebook's introduction and adapt the UCI data-acquisition example on 21 October 2026, following the authors' instructions. The authors should review the wording, run and understand the code, and update this acknowledgment to reflect any later assistance before submission.